# Regime Validation
Validate detected regimes against known historical stress episodes embedded in the synthetic data.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..")))

from src.data.synthetic_generator import generate_credit_data, REGIME_NAMES
from src.data.feature_builder import build_features, standardise_features, get_observation_matrix
from src.models.hmm_model import CreditHMM
from src.models.viterbi import viterbi_from_model
from src.evaluation.metrics import regime_persistence, classification_accuracy, detection_latency, brier_score

In [ ]:
df, true_states = generate_credit_data(n_days=3500, seed=42)
feat = build_features(df)
n_train = int(len(feat) * 0.75)
feat_train_s, feat_test_s, scaler = standardise_features(feat.iloc[:n_train], feat.iloc[n_train:])
X_all = get_observation_matrix(pd.concat([feat_train_s, feat_test_s]))
true_aligned = true_states[df.index.isin(feat.index)]

In [ ]:
hmm = CreditHMM(n_components=3, n_iter=150, n_init=5, random_state=42)
hmm.fit(get_observation_matrix(feat_train_s))
viterbi_states, vlog = viterbi_from_model(hmm, X_all)
hmm_probs = hmm.predict_proba(X_all)
print(f"Viterbi log-prob: {vlog:.2f}")

## Regime Persistence Statistics

In [ ]:
persist_df = regime_persistence(viterbi_states)
print("Viterbi decoded regimes:")
print(persist_df.to_string())

In [ ]:
persist_true = regime_persistence(true_aligned)
print("True regimes (ground truth):")
print(persist_true.to_string())

## Classification Accuracy

In [ ]:
acc = classification_accuracy(true_aligned, viterbi_states)
print(f"Accuracy: {acc['accuracy']:.1%}")
print(acc['report'])

In [ ]:
cm = acc['confusion_matrix']
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax)
ax.set_title("Confusion Matrix: Viterbi vs True Regimes")
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

## Brier Score & Detection Latency

In [ ]:
bs = brier_score(true_aligned, hmm_probs)
lat = detection_latency(true_aligned, viterbi_states, target_regime=2)
print(f"Brier Score: {bs:.4f}")
print(f"Stress Detection Latency: {lat:.1f} days")

In [ ]:
# Regime overlay on CDX IG
feat_idx = pd.concat([feat_train_s, feat_test_s]).index
ig_series = df.loc[feat_idx, "cdx_ig"]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
colors = {0: "#2ecc71", 1: "#3498db", 2: "#e74c3c"}

for ax, states_arr, title in [
    (ax1, viterbi_states, "Viterbi Decoded"),
    (ax2, true_aligned, "True Labels")
]:
    ax.plot(feat_idx, ig_series.values, color="black", lw=1, label="CDX IG")
    for t in range(len(feat_idx) - 1):
        ax.axvspan(feat_idx[t], feat_idx[t + 1], alpha=0.15, color=colors[states_arr[t]])
    ax.set_title(f"CDX IG Spread with Regime Overlay ({title})")
    ax.set_ylabel("Spread (bps)")

plt.tight_layout()
plt.savefig("regime_overlay.png", dpi=150, bbox_inches="tight")
plt.show()